# Tickets de soporte: limpieza, tokenización, TF-IDF y embeddings

1. Descarga y carga del dataset
2. Limpieza (body vacío y tickets en alemán)
3. Preprocesado y tokenización (oraciones, normalización, tokens) + split train/test
4. Vocabulario (clase `Vocabulary`, con token `<UNK>` y corte por frecuencia)
5. Representación TF-IDF
6. Top palabras global y por cola
7. Word embeddings (GloVe) y análisis de palabras sin vector (OOV)
8. Exportación estructurada (JSONL + manifest)

In [1]:
import json
import os
import re
import string
from collections import Counter
from pathlib import Path

import gensim.downloader as api
import joblib
import numpy as np
import pandas as pd
from datasets import load_dataset, load_from_disk
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from spacy.lang.en.stop_words import STOP_WORDS as STOP_EN

C:\Users\Usuario\proyectonlp\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración y carga de datos

In [2]:
PATH = (os.path.join("..","data", "tickets"))
CSV_PATH = os.path.join(PATH, "raw/dataraw.csv")
PROCESSED_PATH = os.path.join(PATH, "dataprocessed/dataprocessed.csv")
OUTDIR = Path(PATH) / "features"
OUTDIR.mkdir(parents=True, exist_ok=True)

In [3]:
if not os.path.exists(os.path.join(PATH, "dataset_dict.json")):
    load_dataset("Tobi-Bueck/customer-support-tickets").save_to_disk(PATH)

if not os.path.exists(CSV_PATH):
    ds = load_from_disk(PATH)
    # SOLO TIENE TRAIN, POR LO QUE TENEMOS QUE HACER NOSOTROS EL SPLIT
    df = ds["train"].to_pandas()
    os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
    df.to_csv(CSV_PATH, index=False)
else:
    df = pd.read_csv(CSV_PATH)

## 2. Limpieza de datos

In [4]:
def limpieza_datos(df: pd.DataFrame, output_path: str = PROCESSED_PATH) -> pd.DataFrame:
    """Elimina las filas con 'body' vacío (NaN), y los tickets escritos en alemán y guarda el resultado en CSV."""
    n_antes = len(df)
    df_limpio = df.dropna(subset=["body"]).reset_index(drop=True)
    df_limpio = df_limpio.drop(df_limpio[df_limpio["language"] == "de"].index)
    print(f"Filas eliminadas: {n_antes - len(df_limpio)} (quedan {len(df_limpio)})")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_limpio.to_csv(output_path, index=False)
    return df_limpio

In [5]:
df = limpieza_datos(df)
df.head()

Filas eliminadas: 33505 (quedan 28260)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN


> Tenemos también las respuestas (`answer`). Para la primera tarea (clasificar el ticket) esa columna **no se usa**: no existe respuesta en el momento de la clasificación.
>
> Lo ideal sería predecir: **queue** (a quién va dirigido), **priority** (importancia) y **tags**.

## 3. Preprocesado y tokenización

Siguiendo el formato de texto que se usa en la práctica de clasificación con Perceptron (Yelp): **texto en minúsculas, con la puntuación separada por espacios**, de forma que `text.split(" ")` da directamente la lista de tokens y la puntuación se descarta con `token not in string.punctuation`. Aquí ese formato lo generamos nosotros a partir del texto crudo, en tres pasos:

0. **Limpieza previa**: el CSV trae los saltos de línea como el texto literal `\n`, etiquetas HTML `<br>` y placeholders de anonimización (`<name>`, `<tel_num>`, `<acc_num>`...). Los `<br>` y `\n` pasan a salto de línea real (así también separan oraciones) y los placeholders se eliminan: no tienen significado y no tienen vector en los embeddings.
1. **División en oraciones**: se parte el texto (`subject` + `body`) en saltos de línea y tras `.`, `!` o `?`. Se hace sobre el texto original, antes de pasar a minúsculas.
2. **Normalización**: URLs, emails y números se sustituyen por los marcadores `<url>`, `<email>` y `<num>` (reducen vocabulario y evitan tokens casi únicos). Después se pasa a minúsculas.
3. **Tokenización** con una expresión regular que mantiene como un solo token:
   - los marcadores entre `<...>` (también los que ya trae el dataset, como `<name>`),
   - las palabras con apóstrofo (`don't`),
   - cada signo de puntuación suelto.

Columnas resultantes:

| columna | contenido |
|---|---|
| `sentences` | lista de oraciones (JSON) |
| `n_sentences`, `n_tokens` | recuentos |
| `text_tok` | tokens con puntuación, separados por espacios (formato Yelp) |
| `tokens` | tokens sin puntuación, separados por espacios (entrada del vocabulario y de TF-IDF) |
| `split` | `train` / `test` |

No se lematiza ni se eliminan stopwords en este paso: las stopwords se filtran más adelante en TF-IDF y se conservan para los embeddings.

In [6]:
PUNCT = set(string.punctuation)

URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
NUM_RE = re.compile(r"\b\d+(?:[.,]\d+)*\b")
HTML_BR_RE = re.compile(r"<br\s*/?>|<br(?=\w)", re.IGNORECASE)
PLACEHOLDER_RE = re.compile(r"<[a-z_]+>")  # <name>, <tel_num>, <acc_num>...
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")
TOKEN_RE = re.compile(r"<[a-z_]+>|\w+(?:'\w+)*|[^\w\s]")


def clean_raw(text: str) -> str:
    """Limpieza previa: \\n literal y <br> -> salto de línea real; se quitan placeholders."""
    text = text.replace("\\n", "\n")
    text = HTML_BR_RE.sub("\n", text)
    return PLACEHOLDER_RE.sub(" ", text)


def split_sentences(text: str) -> list[str]:
    """Divide un texto en oraciones (saltos de línea y tras . ! ?)."""
    return [s.strip() for s in SENT_SPLIT_RE.split(text) if s and s.strip()]


def normalize(sentence: str) -> str:
    """Sustituye URLs, emails y números por marcadores y pasa a minúsculas."""
    s = URL_RE.sub(" <url> ", sentence)
    s = EMAIL_RE.sub(" <email> ", s)
    s = NUM_RE.sub(" <num> ", s)
    return s.lower()


def tokenize(sentence: str) -> list[str]:
    """Tokeniza una oración: palabras, marcadores <...> y puntuación suelta."""
    return TOKEN_RE.findall(normalize(sentence))

In [7]:
df = df.reset_index(drop=True)

# Un texto por ticket: asunto + cuerpo (NO se usa 'answer': no existe al clasificar)
raw_text = (df["subject"].fillna("") + "\n" + df["body"]).str.strip().map(clean_raw)

sentences = [split_sentences(t) for t in raw_text]
sent_tokens = [[tokenize(s) for s in sents] for sents in sentences]

df["sentences"] = [json.dumps(s, ensure_ascii=False) for s in sentences]
df["n_sentences"] = [len(s) for s in sentences]
df["text_tok"] = [" ".join(t for sent in st for t in sent) for st in sent_tokens]
df["tokens"] = [" ".join(t for sent in st for t in sent if t not in PUNCT) for st in sent_tokens]
df["n_tokens"] = df["tokens"].str.split().str.len()

# Ejemplo de la transformación
i = 0
print("RAW        :", raw_text[i][:300].replace("\n", " | "))
print("ORACIONES  :", sentences[i][:3])
print("TEXT_TOK   :", df.loc[i, "text_tok"][:300])
print("TOKENS     :", df.loc[i, "tokens"][:300])
print()
print(df[["n_sentences", "n_tokens"]].describe().round(1))

RAW        : Account Disruption | Dear Customer Support Team, |  | I am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in mul
ORACIONES  : ['Account Disruption', 'Dear Customer Support Team,', 'I am writing to report a significant problem with the centralized account management portal, which currently appears to be offline.']
TEXT_TOK   : account disruption dear customer support team , i am writing to report a significant problem with the centralized account management portal , which currently appears to be offline . this outage is blocking access to account settings , leading to substantial inconvenience . i have attempted to log in
TOKENS     : account disruption dear customer support team i am writing to report a significant problem with the centralized account management portal which current

In [8]:
# El dataset solo trae 'train', así que el split lo hacemos aquí (antes de construir el vocabulario)
idx_train, idx_test = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=df["queue"],  # quita esta línea si alguna cola tiene muy pocos tickets
)
df["split"] = "train"
df.loc[idx_test, "split"] = "test"
print(df["split"].value_counts())

# Guardamos el CSV procesado (ahora con tokens y split)
df.to_csv(PROCESSED_PATH, index=False)

split
train    22608
test      5652
Name: count, dtype: int64


## 4. Vocabulario

Clase `Vocabulary` basada en la de la práctica (Rao & McMahan, 2019): mapea token ↔ índice y reserva un token `<UNK>` para las palabras desconocidas. Se construye **solo con el train** (evita fuga de información) y con un **corte por frecuencia**: los tokens con menos apariciones que `MIN_FREQ` se representan como `<UNK>`. Se añade `to_serializable` / `from_serializable` para guardarlo en JSON y reutilizarlo.

In [9]:
# Basado en Rao, D., & McMahan, B. (2019). Natural language processing with PyTorch.
class Vocabulary(object):
    """Clase para procesar texto y extraer el vocabulario existente para su posterior mapeo."""

    def __init__(self, token_to_idx=None, add_unk=True, unk_token="<UNK>"):
        """
        Args: token_to_idx (dict): mapa token -> índice preexistente.
            : add_unk (bool): si se añade el token UNK.
            : unk_token (str): el token UNK.
        """
        if token_to_idx is None:
            token_to_idx = {}
        self._token_to_idx = token_to_idx
        self._idx_to_token = {idx: token for token, idx in self._token_to_idx.items()}
        self._add_unk = add_unk
        self._unk_token = unk_token
        self.unk_index = -1
        if add_unk:
            self.unk_index = self.add_token(unk_token)

    def to_serializable(self):
        return {"token_to_idx": self._token_to_idx,
                "add_unk": self._add_unk,
                "unk_token": self._unk_token}

    @classmethod
    def from_serializable(cls, contents):
        return cls(**contents)

    def add_token(self, token):
        """Añade un token (si no existe) y devuelve su índice."""
        if token in self._token_to_idx:
            index = self._token_to_idx[token]
        else:
            index = len(self._token_to_idx)
            self._token_to_idx[token] = index
            self._idx_to_token[index] = token
        return index

    def add_many(self, tokens):
        return [self.add_token(token) for token in tokens]

    def lookup_token(self, token):
        """Índice del token; si no existe devuelve el de <UNK>."""
        if self.unk_index >= 0:
            return self._token_to_idx.get(token, self.unk_index)
        return self._token_to_idx[token]

    def lookup_index(self, index):
        if index not in self._idx_to_token:
            raise KeyError("the index (%d) is not in the Vocabulary" % index)
        return self._idx_to_token[index]

    def __str__(self):
        return "<Vocabulary (size=%d)>" % len(self)

    def __len__(self):
        return len(self._token_to_idx)

In [10]:
MIN_FREQ = 2  # corte por frecuencia: tokens con menos apariciones en train -> <UNK>

train_tokens = [t for toks in df.loc[df["split"] == "train", "tokens"] for t in toks.split()]
word_counts = Counter(train_tokens)

vocab = Vocabulary(add_unk=True, unk_token="<UNK>")
for word, count in word_counts.most_common():  # ordenado por frecuencia
    if count >= MIN_FREQ:
        vocab.add_token(word)

print(vocab)
print(f"Tipos distintos en train: {len(word_counts)} -> en el vocabulario: {len(vocab)}")

# Cobertura sobre test: qué fracción de tokens cae en <UNK>
test_tokens = [t for toks in df.loc[df["split"] == "test", "tokens"] for t in toks.split()]
n_unk = sum(vocab.lookup_token(t) == vocab.unk_index for t in test_tokens)
print(f"Tokens de test desconocidos (<UNK>): {n_unk}/{len(test_tokens)} ({n_unk / len(test_tokens):.2%})")

# Guardamos el vocabulario para reutilizarlo
with open(OUTDIR / "vocab.json", "w", encoding="utf-8") as fp:
    json.dump(vocab.to_serializable(), fp, ensure_ascii=False)

<Vocabulary (size=4891)>
Tipos distintos en train: 5914 -> en el vocabulario: 4891
Tokens de test desconocidos (<UNK>): 550/344663 (0.16%)


## 5. Features TF-IDF

In [11]:
# Usamos los tokens ya procesados (sin puntuación, en minúsculas): el vectorizador
# solo hace split por espacios, no vuelve a tokenizar ni a pasar a minúsculas.
data = df
corpus = data["tokens"].fillna("").to_numpy()
idx_train = np.where(data["split"] == "train")[0]
idx_test = np.where(data["split"] == "test")[0]

vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    token_pattern=None,
    stop_words=sorted(STOP_EN),  # solo inglés: los tickets en alemán ya se eliminaron
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2),
)

# Ajustar SOLO con train para evitar fuga de información
X_train = vectorizer.fit_transform(corpus[idx_train])
X_test = vectorizer.transform(corpus[idx_test])

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Vocabulario: {len(vectorizer.get_feature_names_out())}")

Train: (22608, 70948), Test: (5652, 70948)
Vocabulario: 70948


In [12]:
# Guardar para reutilizar
sparse.save_npz(OUTDIR / "X_train.npz", X_train)
sparse.save_npz(OUTDIR / "X_test.npz", X_test)
np.save(OUTDIR / "idx_train.npy", idx_train)  # para recuperar queue/priority/tags
np.save(OUTDIR / "idx_test.npy", idx_test)
joblib.dump(vectorizer, OUTDIR / "tfidf_vectorizer.joblib")

['..\\data\\tickets\\features\\tfidf_vectorizer.joblib']

## 6. Top palabras

In [13]:
TOP_N = 5
feature_names = vectorizer.get_feature_names_out()

# 1) Top global: peso TF-IDF medio por término en train
mean_tfidf = np.asarray(X_train.mean(axis=0)).ravel()
top_global = np.argsort(mean_tfidf)[::-1][:TOP_N]

print(f"Top {TOP_N} palabras (TF-IDF medio, global):")
for i in top_global:
    print(f"  {feature_names[i]:<30} {mean_tfidf[i]:.4f}")

Top 5 palabras (TF-IDF medio, global):
  data                           0.0327
  issue                          0.0196
  security                       0.0190
  medical                        0.0179
  support                        0.0175


In [14]:
# 2) Top por cola (queue): TF-IDF medio solo con los tickets de esa cola
y_train = data["queue"].iloc[idx_train].to_numpy()
top_por_cola = {}

for cola in np.unique(y_train):
    mask = y_train == cola
    media_cola = np.asarray(X_train[mask].mean(axis=0)).ravel()
    top_idx = np.argsort(media_cola)[::-1][:TOP_N]
    top_por_cola[cola] = [(feature_names[i], media_cola[i]) for i in top_idx]

    print(f"\nTop {TOP_N} palabras para la cola '{cola}':")
    for palabra, peso in top_por_cola[cola]:
        print(f"  {palabra:<30} {peso:.4f}")


Top 5 palabras para la cola 'Billing and Payments':
  billing                        0.0736
  payment                        0.0439
  information                    0.0247
  data                           0.0225
  services                       0.0221

Top 5 palabras para la cola 'Customer Service':
  data                           0.0360
  investment                     0.0209
  analytics                      0.0196
  security                       0.0196
  information                    0.0195

Top 5 palabras para la cola 'General Inquiry':
  data                           0.0399
  investment                     0.0251
  engagement                     0.0240
  digital                        0.0233
  strategies                     0.0213

Top 5 palabras para la cola 'Human Resources':
  data                           0.0341
  software                       0.0226
  employee                       0.0224
  security                       0.0203
  issue                          0.0183

T

## 7. Word embeddings (GloVe) y palabras sin vector (OOV)

**Modelo:** GloVe `glove-wiki-gigaword-100` (gensim): 400.000 palabras, 100 dimensiones, todo en minúsculas (como nuestros tokens). La primera vez descarga ~130 MB a la caché de gensim.

**Adaptar los tokens a GloVe:** GloVe separa las contracciones (`don't` → `do` + `n't`), pero nuestro tokenizador las deja como una sola palabra, y así `don't` saldría como OOV sin serlo realmente. Por eso, solo para buscar el vector, las contracciones se parten igual que GloVe, y los marcadores `<num>`, `<url>`, `<email>` se ignoran (no son palabras).

**Vector por ticket:** media de los vectores de sus palabras (las OOV se ignoran).

In [15]:
GLOVE = "glove-wiki-gigaword-100"
kv = api.load(GLOVE)  # descarga la primera vez
print(f"{GLOVE}: {len(kv)} palabras, {kv.vector_size} dimensiones")

MARKER_RE = re.compile(r"<[a-z_]+>")


def glove_tokens(tokens: list[str]) -> list[str]:
    """Tokens del ticket tal y como los espera GloVe (contracciones separadas, sin marcadores)."""
    out = []
    for t in tokens:
        if MARKER_RE.fullmatch(t):
            continue
        if t.endswith("n't") and len(t) > 3:
            out += [t[:-3], "n't"]
        elif "'" in t and not t.startswith("'"):
            base, suf = t.split("'", 1)
            out += [base, "'" + suf]
        else:
            out.append(t)
    return out


emb_tokens = [glove_tokens(t.split()) for t in df["tokens"]]

glove-wiki-gigaword-100: 400000 palabras, 100 dimensiones


In [16]:
# Análisis de OOV: palabras (tipos) y apariciones (tokens) sin vector
freq = Counter(t for toks in emb_tokens for t in toks)
oov = {w: c for w, c in freq.items() if w not in kv}

n_tipos, n_oov_tipos = len(freq), len(oov)
n_tokens, n_oov_tokens = sum(freq.values()), sum(oov.values())
print(f"Palabras distintas: {n_tipos} | sin vector: {n_oov_tipos} ({n_oov_tipos / n_tipos:.1%})")
print(f"Tokens totales:     {n_tokens} | sin vector: {n_oov_tokens} ({n_oov_tokens / n_tokens:.1%})")

oov_df = (pd.DataFrame(oov.items(), columns=["word", "count"])
          .sort_values("count", ascending=False).reset_index(drop=True))
EMB_DIR = Path(PATH) / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)
oov_df.to_csv(EMB_DIR / "oov_words.csv", index=False)
print("\nTop 20 OOV más frecuentes:")
print(oov_df.head(20).to_string(index=False))

Palabras distintas: 5955 | sin vector: 247 (4.1%)
Tokens totales:     1716336 | sin vector: 7763 (0.5%)

Top 20 OOV más frecuentes:
             word  count
          clickup    553
    elasticsearch    379
       smartsheet    329
            redis    325
         airtable    278
           zapier    247
       kubernetes    244
    misconfigured    208
          alteryx    203
 misconfiguration    188
       rapidminer    181
        datarobot    177
          magento    147
         sendgrid    137
     malwarebytes    124
   activecampaign    122
        mailchimp    120
      woocommerce    120
          pytorch    117
misconfigurations    115


**Conclusión del análisis:** el número de palabras sin vector **no es grande** (ver porcentajes arriba): casi todo el texto queda cubierto. Las OOV son sobre todo nombres de producto o tecnología (`clickup`, `elasticsearch`, `kubernetes`...), que GloVe, entrenado con Wikipedia y noticias, casi no ha visto. Por eso entrenar un Word2Vec propio no es imprescindible; su ventaja sería cubrir esas marcas. *(Decisión pendiente de confirmar por el equipo.)*

In [17]:
# Vector por ticket = media de los vectores de sus palabras (las OOV se ignoran)
ticket_vectors = np.zeros((len(df), kv.vector_size), dtype=np.float32)
sin_vector = 0
for i, toks in enumerate(emb_tokens):
    vs = [kv[t] for t in toks if t in kv]
    if vs:
        ticket_vectors[i] = np.mean(vs, axis=0)
    else:
        sin_vector += 1

np.save(EMB_DIR / "ticket_vectors.npy", ticket_vectors)
print(f"Vectores de tickets: {ticket_vectors.shape} | tickets sin ninguna palabra con vector: {sin_vector}")

Vectores de tickets: (28260, 100) | tickets sin ninguna palabra con vector: 2


## 8. Exportación estructurada

Todo el texto procesado queda en **un JSONL (un ticket por línea)** con `id`, `split`, `text`, `sentences`, `tokens`, `queue`, `priority`, `type` y `tags`. Se eligió JSONL porque admite listas (oraciones, tokens, tags) dentro de cada registro, cosa que un CSV no representa bien, y se lee línea a línea.

Las matrices (TF-IDF y embeddings) **no** van en el JSON porque son numéricas y grandes: se guardan aparte (`.npz` / `.npy`) y se alinean por posición: la fila *i* de cada matriz es el ticket con `id == i`. El `manifest.json` explica cada archivo.

In [18]:
EXPORT_DIR = Path(PATH) / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
TAG_COLS = [f"tag_{i}" for i in range(1, 9)]

with (EXPORT_DIR / "tickets_processed.jsonl").open("w", encoding="utf-8") as f:
    for i, fila in enumerate(df.itertuples(index=False)):
        reg = {
            "id": i,
            "split": fila.split,
            "text": raw_text[i],
            "sentences": sentences[i],
            "tokens": fila.tokens.split(),
            "n_sentences": fila.n_sentences,
            "n_tokens": fila.n_tokens,
            # Etiquetas a predecir
            "queue": fila.queue,
            "priority": fila.priority,
            "type": fila.type,
            "tags": [t for t in (getattr(fila, c) for c in TAG_COLS) if isinstance(t, str)],
        }
        f.write(json.dumps(reg, ensure_ascii=False) + "\n")

manifest = {
    "n_tickets": len(df),
    "n_train": int((df["split"] == "train").sum()),
    "n_test": int((df["split"] == "test").sum()),
    "archivos": {
        "export/tickets_processed.jsonl": "Un ticket por línea: id, split, text, sentences, tokens, queue, priority, type, tags. Leer con pd.read_json(..., lines=True).",
        "features/X_train.npz, X_test.npz": "Matrices TF-IDF dispersas (sparse.load_npz). Filas = tickets de idx_train / idx_test.",
        "features/idx_train.npy, idx_test.npy": "Valores de 'id' de cada split, en el orden de las filas de X_train / X_test.",
        "features/tfidf_vectorizer.joblib": "TfidfVectorizer ajustado con train (joblib.load).",
        "features/vocab.json": "Vocabulario (token <-> índice, con <UNK>) construido con train. Vocabulary.from_serializable.",
        "embeddings/ticket_vectors.npy": "Matriz (n_tickets x 100): media de los vectores GloVe de cada ticket. Fila i = ticket con id i.",
        "embeddings/oov_words.csv": "Palabras de los tickets sin vector en GloVe y su frecuencia.",
    },
}
(EXPORT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{len(df)} tickets exportados -> {EXPORT_DIR / 'tickets_processed.jsonl'}")

28260 tickets exportados -> ..\data\tickets\export\tickets_processed.jsonl
